# Diffusion No-Context Forecasting

<!-- Commented sorted notebook -->
Purpose: train and evaluate the cleaned supervisor-facing version of this generative forecaster. This version receives known future/static covariates only; the past kWh context is intentionally removed.

The notebook follows the shared experiment structure: setup, preprocessing, batching, model training, checkpointing, validation sampling, and evaluation export.


In [ ]:

import math
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)

MODEL_LABEL = "Diffusion-NoContext-Forecaster"
MODEL_TAG = "forecast-nocontext-diffusion"

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

FORECAST_HORIZONS = {"24h": pd.Timedelta("24h"), "7d": pd.Timedelta("7d"), "28d": pd.Timedelta("28d")}
ALLOWED_FREQS = {"30min", "1H", "2H", "3H", "6H"}
FREQ = "30min"
PRIMARY_HORIZON = "24h"

MAX_CONTEXT_LEN = 336
CONTEXT_MULTIPLIER = 0
WINDOW_STRIDE_MULTIPLIER = 1

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
LR = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


epochs_diff = 500
hidden_dim = 128
cond_dim = 32
latent_dim = 32
grad_clip = 1.0


diffusion_steps = 300
beta_start = 1e-4
beta_end = 0.02


sample_temperature = 0.8



STATIC_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]

CATEGORICAL_COLS = ["StationNo", "TRIAL_REGION_NAME"]

RAW_TIME_COV_COLS = [
    "Temperature",
    "Weekday",
    "Month",
    "Hour",
    "CDD",
    "HDD",
    "wind_speed",
    "temperature_lag_2",
    "temperature_lag_6",
    "temperature_lag_48",
    "temperature_lag_144",
]

TIME_COV_COLS = [
    "Temperature",
    "Hour_sin",
    "Hour_cos",
    "Weekday_sin",
    "Weekday_cos",
    "Month_sin",
    "Month_cos",
    "CDD",
    "HDD",
    "wind_speed",
    "temperature_lag_2",
    "temperature_lag_6",
    "temperature_lag_48",
    "temperature_lag_144",
]

# Keeps pandas frequency strings explicit so 30min/1H/2H/3H experiments resample consistently.
def pandas_freq(freq: str) -> str:
    return freq.replace("H", "h")

# Converts a human horizon label into target-window steps at the selected data frequency.
def seq_len_for_horizon(primary_horizon, freq):
    if primary_horizon not in FORECAST_HORIZONS:
        raise ValueError(f"Unsupported horizon {primary_horizon}. Use one of {sorted(FORECAST_HORIZONS)}")
    if freq not in ALLOWED_FREQS:
        raise ValueError(f"Unsupported freq {freq}. Use one of {sorted(ALLOWED_FREQS)}")
    steps = FORECAST_HORIZONS[primary_horizon] / pd.to_timedelta(pandas_freq(freq))
    if not float(steps).is_integer():
        raise ValueError(f"Horizon {primary_horizon} is not divisible by freq {freq}")
    return int(steps)

SEQ_LEN = seq_len_for_horizon(PRIMARY_HORIZON, FREQ)
CONTEXT_LEN = min(CONTEXT_MULTIPLIER * SEQ_LEN, MAX_CONTEXT_LEN)
WINDOW_STRIDE = max(1, WINDOW_STRIDE_MULTIPLIER * SEQ_LEN)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("medium")

print(f"{MODEL_LABEL} | FREQ={FREQ} | horizon={PRIMARY_HORIZON} | context={CONTEXT_LEN} | horizon_steps={SEQ_LEN} | stride={WINDOW_STRIDE} | device={DEVICE}")

In [ ]:

# Searches nearby project folders for the cleaned household energy dataset used by all notebooks.
def find_data_file():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base / "data_with_weather.pickle",
            base / "models" / "data_with_weather.pickle",
            base / "data_min.parquet",
            base / "models" / "data_min.parquet",
        ])
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find data_with_weather.pickle or data_min.parquet from the notebook location.")

# Cyclical sin/cos encoding preserves periodic hour/weekday/month distance; source: https://feature-engine.trainindata.com/en/latest/user_guide/creation/CyclicalFeatures.html
def add_cyclical_time_features(frame, datetime_values):
    dt = pd.DatetimeIndex(pd.to_datetime(datetime_values))
    hour = dt.hour.astype(np.float32)
    minute = dt.minute.astype(np.float32)
    hour_float = hour + minute / 60.0
    weekday = dt.weekday.astype(np.float32)
    month = dt.month.astype(np.float32)

    frame["Hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Weekday_sin"] = np.sin(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Weekday_cos"] = np.cos(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Month_sin"] = np.sin(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Month_cos"] = np.cos(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Hour"] = hour.astype("float32")
    frame["Weekday"] = weekday.astype("float32")
    frame["Month"] = month.astype("float32")
    return frame

data_path = find_data_file()
if data_path.suffix == ".pickle":
    df = pd.read_pickle(data_path)
else:
    df = pd.read_parquet(data_path)

needed = [TIME_COL, TARGET_COL, HOUSEHOLD_ID_COL] + STATIC_COLS + CATEGORICAL_COLS + RAW_TIME_COV_COLS
for col in needed:
    if col not in df.columns:
        df[col] = 0.0 if col not in CATEGORICAL_COLS else "missing"
df = df[needed].copy()

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
df = df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])
add_cyclical_time_features(df, df[TIME_COL])

for col in STATIC_COLS + RAW_TIME_COV_COLS + [TARGET_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).astype("float32")

if "Temperature" in df.columns:
    temp = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
    df["CDD"] = np.maximum(temp - 18.0, 0.0).astype("float32")
    df["HDD"] = np.maximum(18.0 - temp, 0.0).astype("float32")

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("string").fillna("missing")
    df[f"{col}_id"], _ = pd.factorize(df[col], sort=True)
    df[f"{col}_id"] = df[f"{col}_id"].astype("int64")

num_stations = int(df["StationNo_id"].max()) + 1
num_regions = int(df["TRIAL_REGION_NAME_id"].max()) + 1

agg = {TARGET_COL: "sum"}
for col in STATIC_COLS + [f"{c}_id" for c in CATEGORICAL_COLS]:
    agg[col] = "first"
for col in RAW_TIME_COV_COLS + TIME_COV_COLS:
    if col in df.columns and col not in agg:
        agg[col] = "mean"

df = (
    df.set_index(TIME_COL)
      .groupby(HOUSEHOLD_ID_COL)
      .resample(pandas_freq(FREQ))
      .agg(agg)
      .reset_index()
      .sort_values([HOUSEHOLD_ID_COL, TIME_COL])
)
add_cyclical_time_features(df, df[TIME_COL])

for col in STATIC_COLS + TIME_COV_COLS + [TARGET_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).astype("float32")
for col in [f"{c}_id" for c in CATEGORICAL_COLS]:
    df[col] = pd.to_numeric(df[col], errors="coerce").ffill().bfill().fillna(0).astype("int64")

print("Dataframe shape:", df.shape)
print("num_stations:", num_stations, "| num_regions:", num_regions)

In [ ]:

rng = np.random.default_rng(SEED)
all_customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
n_val = max(1, int(len(all_customers) * VAL_FRAC))
val_customers = set(rng.choice(all_customers, size=n_val, replace=False).tolist())
print(f"Customers total={len(all_customers)} | train={len(all_customers)-len(val_customers)} | val={len(val_customers)}")

def empty_float(*shape):
    return np.empty(shape, dtype=np.float32)

def empty_int(*shape):
    return np.empty(shape, dtype=np.int64)

y_past_parts, y_future_parts = [], []
x_time_past_parts, x_time_future_parts = [], []
x_static_parts, x_static_cat_parts = [], []
y_past_val_parts, y_future_val_parts = [], []
x_time_past_val_parts, x_time_future_val_parts = [], []
x_static_val_parts, x_static_cat_val_parts = [], []

for household_id, g in df.groupby(HOUSEHOLD_ID_COL):
    g = g.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL], keep="last")
    if g.empty:
        continue

    idx = pd.date_range(g[TIME_COL].min(), g[TIME_COL].max(), freq=pandas_freq(FREQ))
    g = g.set_index(TIME_COL).reindex(idx)
    g.index.name = TIME_COL
    g[HOUSEHOLD_ID_COL] = household_id

    for col in STATIC_COLS:
        g[col] = pd.to_numeric(g[col], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    for col in [f"{c}_id" for c in CATEGORICAL_COLS]:
        g[col] = pd.to_numeric(g[col], errors="coerce").ffill().bfill().fillna(0).astype("int64")

    add_cyclical_time_features(g, g.index)
    g[TARGET_COL] = pd.to_numeric(g[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
    for col in TIME_COV_COLS:
        g[col] = pd.to_numeric(g[col], errors="coerce").fillna(0.0).astype("float32")

    g = g.reset_index()
    if len(g) < CONTEXT_LEN + SEQ_LEN:
        continue

    y_all = g[[TARGET_COL]].to_numpy(dtype=np.float32)
    x_time_all = g[TIME_COV_COLS].to_numpy(dtype=np.float32)
    x_static_vec = g[STATIC_COLS].iloc[0].to_numpy(dtype=np.float32)
    x_cat_vec = g[[f"{c}_id" for c in CATEGORICAL_COLS]].iloc[0].to_numpy(dtype=np.int64)

    is_val = household_id in val_customers
    yp_list = y_past_val_parts if is_val else y_past_parts
    yf_list = y_future_val_parts if is_val else y_future_parts
    xtp_list = x_time_past_val_parts if is_val else x_time_past_parts
    xtf_list = x_time_future_val_parts if is_val else x_time_future_parts
    xs_list = x_static_val_parts if is_val else x_static_parts
    xc_list = x_static_cat_val_parts if is_val else x_static_cat_parts

    max_start = len(g) - CONTEXT_LEN - SEQ_LEN
    for start in range(0, max_start + 1, WINDOW_STRIDE):
        mid = start + CONTEXT_LEN
        end = mid + SEQ_LEN
        yp_list.append(y_all[start:mid])
        yf_list.append(y_all[mid:end])
        xtp_list.append(x_time_all[start:mid])
        xtf_list.append(x_time_all[mid:end])
        xs_list.append(x_static_vec)
        xc_list.append(x_cat_vec)

y_past = np.stack(y_past_parts).astype(np.float32)
y_future = np.stack(y_future_parts).astype(np.float32)
x_time_past = np.stack(x_time_past_parts).astype(np.float32)
x_time_future = np.stack(x_time_future_parts).astype(np.float32)
x_static = np.stack(x_static_parts).astype(np.float32)
static_cat_ids = np.stack(x_static_cat_parts).astype(np.int64)

y_past_val = np.stack(y_past_val_parts).astype(np.float32)
y_future_val = np.stack(y_future_val_parts).astype(np.float32)
x_time_past_val = np.stack(x_time_past_val_parts).astype(np.float32)
x_time_future_val = np.stack(x_time_future_val_parts).astype(np.float32)
x_static_val = np.stack(x_static_val_parts).astype(np.float32)
static_cat_ids_val = np.stack(x_static_cat_val_parts).astype(np.int64)

print("TRAIN y_past:", y_past.shape, "y_future:", y_future.shape, "x_time_past:", x_time_past.shape, "x_time_future:", x_time_future.shape)
print("VAL   y_past:", y_past_val.shape, "y_future:", y_future_val.shape, "x_time_past:", x_time_past_val.shape, "x_time_future:", x_time_future_val.shape)

y_scaler = MinMaxScaler(feature_range=(0, 1))
y_train_log_flat = np.concatenate([
    np.log1p(y_past).reshape(-1, 1),
    np.log1p(y_future).reshape(-1, 1),
], axis=0)
y_scaler.fit(y_train_log_flat)

# Applies the target scaler only to kWh values so generated outputs can be inverted back to physical units.
def scale_y(arr):
    if arr.size == 0:
        return arr.astype(np.float32)
    return y_scaler.transform(np.log1p(arr).reshape(-1, 1)).reshape(arr.shape).astype(np.float32)

# Converts scaled model outputs back to kWh for metrics and plots.
def y_scaled_to_kwh(arr_scaled):
    arr_log = y_scaler.inverse_transform(np.asarray(arr_scaled).reshape(-1, 1)).reshape(np.asarray(arr_scaled).shape)
    return np.maximum(np.expm1(arr_log), 0.0).astype(np.float32)

y_past_scaled = scale_y(y_past)
y_future_scaled = scale_y(y_future)
y_past_scaled_val = scale_y(y_past_val)
y_future_scaled_val = scale_y(y_future_val)

c_dim = x_time_future.shape[2]
c_scaler = StandardScaler()
c_scaler.fit(np.concatenate([
    x_time_past.reshape(-1, c_dim),
    x_time_future.reshape(-1, c_dim),
], axis=0))
# Reuses the fitted time-covariate scaler when no-context notebooks rebuild the zero-context tensors.
def scale_time(arr):
    if arr.size == 0:
        return arr.astype(np.float32)
    return c_scaler.transform(arr.reshape(-1, c_dim)).reshape(arr.shape).astype(np.float32)

x_time_past_scaled = scale_time(x_time_past)
x_time_future_scaled = scale_time(x_time_future)
x_time_past_scaled_val = scale_time(x_time_past_val)
x_time_future_scaled_val = scale_time(x_time_future_val)

s_scaler = StandardScaler()
x_static_scaled = s_scaler.fit_transform(x_static).astype(np.float32)
x_static_scaled_val = s_scaler.transform(x_static_val).astype(np.float32)

# Packs past context, future target, temporal covariates, and static covariates into one training item.
class EnergyForecastDataset(Dataset):
    def __init__(self, y_past, y_future, x_time_past, x_time_future, x_static_num, x_static_cat):
        self.y_past = torch.from_numpy(y_past)
        self.y_future = torch.from_numpy(y_future)
        self.x_time_past = torch.from_numpy(x_time_past)
        self.x_time_future = torch.from_numpy(x_time_future)
        self.x_static_num = torch.from_numpy(x_static_num)
        self.x_static_cat = torch.from_numpy(x_static_cat).long()

    def __len__(self):
        return self.y_future.shape[0]

    def __getitem__(self, idx):
        return (
            self.y_past[idx],
            self.y_future[idx],
            self.x_time_past[idx],
            self.x_time_future[idx],
            self.x_static_num[idx],
            self.x_static_cat[idx],
        )

dataset = EnergyForecastDataset(
    y_past_scaled, y_future_scaled, x_time_past_scaled, x_time_future_scaled,
    x_static_scaled, static_cat_ids
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, pin_memory=torch.cuda.is_available())

y_dim = y_future_scaled.shape[2]
c_dim = x_time_future_scaled.shape[2]
s_dim = x_static_scaled.shape[1]

past_min = float(y_past_scaled.min()) if y_past_scaled.size else float("nan")
past_max = float(y_past_scaled.max()) if y_past_scaled.size else float("nan")
print("scaled ranges:", past_min, past_max, y_future_scaled.min(), y_future_scaled.max())
print("dims:", "y_dim", y_dim, "c_dim", c_dim, "s_dim", s_dim)

In [ ]:

# Combines future time covariates, static numeric features, categorical embeddings, and optional past context.
class FutureConditionProjector(nn.Module):
    def __init__(self, c_dim, s_num_dim, past_dim, cond_dim, num_stations, num_regions, station_emb_dim=16, region_emb_dim=4):
        super().__init__()
        self.station_emb = nn.Embedding(num_stations, station_emb_dim)
        self.region_emb = nn.Embedding(num_regions, region_emb_dim)
        in_dim = c_dim + s_num_dim + station_emb_dim + region_emb_dim + past_dim
        self.proj = nn.Sequential(
            nn.Linear(in_dim, cond_dim),
            nn.LayerNorm(cond_dim),
            nn.Tanh(),
        )

    def forward(self, x_time_future, x_static_num, x_static_cat, past_context):
        B, T, _ = x_time_future.shape
        station_id = x_static_cat[:, 0].long()
        region_id = x_static_cat[:, 1].long()
        st = self.station_emb(station_id).unsqueeze(1).expand(B, T, -1)
        rg = self.region_emb(region_id).unsqueeze(1).expand(B, T, -1)
        xs = x_static_num.unsqueeze(1).expand(B, T, -1)
        pc = past_context.unsqueeze(1).expand(B, T, -1)
        return self.proj(torch.cat([x_time_future, xs, st, rg, pc], dim=-1))
# Embeds the diffusion noise step, separate from the physical timestep inside the energy window; source: https://arxiv.org/abs/2006.11239
class TimeEmbedding(nn.Module):
    def __init__(self, steps, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -(math.log(10000.0) / max(1, half - 1)))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if emb.shape[1] < self.dim:
            emb = F.pad(emb, (0, self.dim - emb.shape[1]))
        return self.mlp(emb)


In [ ]:

# DDPM-style denoiser learns to predict injected Gaussian noise under conditions; source: https://arxiv.org/abs/2006.11239
# Implementation reference for DDPM training/sampling structure: https://github.com/openai/improved-diffusion
class ForecastDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.cond = FutureConditionProjector(c_dim, s_dim, hidden_dim, cond_dim, num_stations, num_regions)
        self.time_emb = TimeEmbedding(diffusion_steps, hidden_dim)
        in_dim = y_dim + cond_dim + hidden_dim
        self.net = nn.LSTM(in_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, y_dim)

    # Builds the conditional tensor reused by training, sampling, and evaluation.
    def build_cond(self, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        past_context = torch.zeros(y_past.shape[0], hidden_dim, device=y_past.device, dtype=y_past.dtype)
        return self.cond(x_time_future, x_static_num, x_static_cat, past_context)

    def forward(self, y_noisy, t, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        cond = self.build_cond(y_past, x_time_past, x_time_future, x_static_num, x_static_cat)
        t_emb = self.time_emb(t).unsqueeze(1).expand(y_noisy.shape[0], y_noisy.shape[1], -1)
        h = torch.cat([y_noisy, cond, t_emb], dim=-1)
        h, _ = self.net(h)
        return self.out(h)


# Linear beta noise schedule follows the original DDPM setup; source: https://arxiv.org/abs/2006.11239
betas = torch.linspace(beta_start, beta_end, diffusion_steps, device=DEVICE)
# alpha_t is the retained signal fraction at diffusion step t; source: https://arxiv.org/abs/2006.11239
alphas = 1.0 - betas
# alpha_bar_t is the cumulative retained signal used in the closed-form forward noising equation; source: https://arxiv.org/abs/2006.11239
alpha_bars = torch.cumprod(alphas, dim=0)

# Forward diffusion noising: y_t = sqrt(alpha_bar_t)y_0 + sqrt(1-alpha_bar_t)eps; source: https://arxiv.org/abs/2006.11239
# Implementation reference for DDPM forward noising utilities: https://github.com/openai/improved-diffusion
def q_sample(y0, t, noise):
    a_bar = alpha_bars[t].view(-1, 1, 1)
    return torch.sqrt(a_bar) * y0 + torch.sqrt(1.0 - a_bar) * noise

model_diff = ForecastDenoiser().to(DEVICE)
opt_diff = optim.AdamW(model_diff.parameters(), lr=LR, weight_decay=1e-4)

for epoch in range(epochs_diff):
    model_diff.train()
    running = 0.0
    for y_past_b, y_future_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b in loader:
        y_past_b = y_past_b.to(DEVICE, non_blocking=True)
        y_future_b = y_future_b.to(DEVICE, non_blocking=True)
        x_time_past_b = x_time_past_b.to(DEVICE, non_blocking=True)
        x_time_future_b = x_time_future_b.to(DEVICE, non_blocking=True)
        x_static_b = x_static_b.to(DEVICE, non_blocking=True)
        x_static_cat_b = x_static_cat_b.to(DEVICE, non_blocking=True).long()

        t = torch.randint(0, diffusion_steps, (y_future_b.shape[0],), device=DEVICE)
        noise = torch.randn_like(y_future_b)
        y_noisy = q_sample(y_future_b, t, noise)
        pred_noise = model_diff(y_noisy, t, y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)
        loss = F.mse_loss(pred_noise, noise)
        opt_diff.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip:
            torch.nn.utils.clip_grad_norm_(model_diff.parameters(), grad_clip)
        opt_diff.step()
        running += loss.item()
    if epoch % 10 == 0:
        print(f"[{MODEL_LABEL}] epoch {epoch:03d} | noise_mse={running/len(loader):.5f}")

active_model = model_diff

@torch.no_grad()
# Generates one batch of probabilistic forecasts in scaled space before kWh inversion.
def generate_one_batch(y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b):
    active_model.eval()
    y = torch.randn(y_past_b.shape[0], SEQ_LEN, y_dim, device=y_past_b.device) * sample_temperature
    for step in reversed(range(diffusion_steps)):
        t = torch.full((y.shape[0],), step, device=y.device, dtype=torch.long)
        eps = active_model(y, t, y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)
        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]
        # DDPM reverse mean step removes predicted noise one schedule step at a time; source: https://arxiv.org/abs/2006.11239
        y = (1.0 / torch.sqrt(alpha_t)) * (y - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps)
        if step > 0:
            y = y + torch.sqrt(beta_t) * torch.randn_like(y) * sample_temperature
    return y.clamp(0.0, 1.0)

In [ ]:

# Saves enough metadata with the weights to identify the exact experiment settings later.
def checkpoint_payload():
    extra = {
        "model_type": MODEL_LABEL,
        "FREQ": FREQ,
        "PRIMARY_HORIZON": PRIMARY_HORIZON,
        "SEQ_LEN": SEQ_LEN,
        "CONTEXT_LEN": CONTEXT_LEN,
        "WINDOW_STRIDE": WINDOW_STRIDE,
        "TIME_COV_COLS": TIME_COV_COLS,
        "STATIC_COLS": STATIC_COLS,
        "CATEGORICAL_COLS": CATEGORICAL_COLS,
        "y_scaler_min": y_scaler.min_.tolist(),
        "y_scaler_scale": y_scaler.scale_.tolist(),
        "num_stations": num_stations,
        "num_regions": num_regions,
        "hidden_dim": hidden_dim,
        "cond_dim": cond_dim,
        "latent_dim": latent_dim,
    }
    return extra

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_name = (
    f"{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}__ctx-{CONTEXT_LEN}__seq-{SEQ_LEN}"
    f"__seed-{SEED}"
)
checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pt"

payload = {"extra": checkpoint_payload(), "model_state_dict": model_diff.state_dict(), "optimizer_state_dict": opt_diff.state_dict()}
torch.save(payload, checkpoint_path)
print("Saved:", checkpoint_path)

In [ ]:

SAMPLES = 200
EVAL_BATCH_SIZE = 64
EVAL_SEED = SEED + 123
EVAL_POOLS = ("validation",)
BINS = 80
CLIP_QUANTILE = None
ALPHA_PI = 0.10

# Dynamic Time Warping compares sequence shape even when peaks are slightly shifted; source: https://en.wikipedia.org/wiki/Dynamic_time_warping
def dtw_distance(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    n, m = len(a), len(b)
    dp = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return float(dp[n, m])

# Autocorrelation checks whether generated windows preserve temporal dependence; source: https://en.wikipedia.org/wiki/Autocorrelation
def autocorr(x, max_lag):
    x = np.asarray(x, dtype=np.float32)
    x = x - x.mean()
    denom = np.dot(x, x) + 1e-12
    out = [1.0]
    for lag in range(1, max_lag + 1):
        out.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.asarray(out)

def mean_acf(arr):
    max_lag = min(200, arr.shape[1] - 1)
    return np.mean([autocorr(arr[i, :, 0], max_lag) for i in range(arr.shape[0])], axis=0)

def window_feature_embed(x):
    y = x[:, :, 0]
    acf_lag = min(3, y.shape[1] - 1)
    acfs = np.stack([autocorr(row, acf_lag) for row in y], axis=0)
    return np.column_stack([
        y.mean(axis=1),
        y.std(axis=1),
        y.min(axis=1),
        y.max(axis=1),
        np.quantile(y, 0.10, axis=1),
        np.quantile(y, 0.50, axis=1),
        np.quantile(y, 0.90, axis=1),
        y.sum(axis=1),
        y.argmax(axis=1) / max(1, y.shape[1] - 1),
        acfs[:, 1] if acfs.shape[1] > 1 else np.zeros(y.shape[0]),
        acfs[:, 2] if acfs.shape[1] > 2 else np.zeros(y.shape[0]),
        acfs[:, 3] if acfs.shape[1] > 3 else np.zeros(y.shape[0]),
    ]).astype(np.float64)

def sqrtm_psd(mat):
    vals, vecs = np.linalg.eigh((mat + mat.T) / 2.0)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T

# Feature Fréchet distance mirrors FID-style Gaussian distance on summary features; source: https://arxiv.org/abs/1706.08500
def frechet_distance(a, b):
    ma, mb = a.mean(axis=0), b.mean(axis=0)
    ca = np.cov(a, rowvar=False) + np.eye(a.shape[1]) * 1e-6
    cb = np.cov(b, rowvar=False) + np.eye(b.shape[1]) * 1e-6
    sqrt_ca = sqrtm_psd(ca)
    covmean = sqrtm_psd(sqrt_ca @ cb @ sqrt_ca)
    return float(np.sum((ma - mb) ** 2) + np.trace(ca + cb - 2 * covmean))

# Pinball loss evaluates quantile forecasts; source: https://en.wikipedia.org/wiki/Quantile_regression
def pinball(y, qhat, q):
    err = y - qhat
    return np.maximum(q * err, (q - 1) * err)

def run_eval_pool(EVAL_POOL):
    rng_eval = np.random.default_rng(EVAL_SEED + (0 if EVAL_POOL == "train" else 10_000))
    if EVAL_POOL == "train":
        n = min(512, len(y_future))
        idx = rng_eval.choice(len(y_future), size=n, replace=False)
        y_past_in = y_past_scaled[idx]
        x_time_past_in = x_time_past_scaled[idx]
        x_time_future_in = x_time_future_scaled[idx]
        x_static_in = x_static_scaled[idx]
        x_static_cat_in = static_cat_ids[idx]
        y_real_kwh = y_future[idx]
    elif EVAL_POOL == "validation":
        n = min(512, len(y_future_val))
        idx = rng_eval.choice(len(y_future_val), size=n, replace=False)
        y_past_in = y_past_scaled_val[idx]
        x_time_past_in = x_time_past_scaled_val[idx]
        x_time_future_in = x_time_future_scaled_val[idx]
        x_static_in = x_static_scaled_val[idx]
        x_static_cat_in = static_cat_ids_val[idx]
        y_real_kwh = y_future_val[idx]
    else:
        raise ValueError(EVAL_POOL)

    PLOT_N = len(y_real_kwh)
    selected_train_windows = PLOT_N if EVAL_POOL == "train" else 0
    selected_val_windows = PLOT_N if EVAL_POOL == "validation" else 0
    DTW_N = min(128, PLOT_N)
    print(f"eval_pool={EVAL_POOL} | windows={PLOT_N} | samples={SAMPLES}")

    y_fake_scaled_samples = []
    active_model.eval()
    with torch.no_grad():
        for s in range(SAMPLES):
            if s % 10 == 0:
                print(f"Generating {MODEL_LABEL} sample {s+1}/{SAMPLES}")
            batches = []
            for start in range(0, PLOT_N, EVAL_BATCH_SIZE):
                end = min(start + EVAL_BATCH_SIZE, PLOT_N)
                ypb = torch.from_numpy(y_past_in[start:end]).to(DEVICE)
                xtpb = torch.from_numpy(x_time_past_in[start:end]).to(DEVICE)
                xtfb = torch.from_numpy(x_time_future_in[start:end]).to(DEVICE)
                xsb = torch.from_numpy(x_static_in[start:end]).to(DEVICE)
                xcb = torch.from_numpy(x_static_cat_in[start:end]).to(DEVICE).long()
                batches.append(generate_one_batch(ypb, xtpb, xtfb, xsb, xcb).cpu().numpy())
            y_fake_scaled_samples.append(np.concatenate(batches, axis=0))

    y_fake_scaled_samples = np.stack(y_fake_scaled_samples, axis=0)
    y_fake_kwh_samples = np.stack([y_scaled_to_kwh(y_fake_scaled_samples[s]) for s in range(SAMPLES)], axis=0)
    y_fake_kwh_point = np.median(y_fake_kwh_samples, axis=0)
    y_fake_kwh_mean = np.mean(y_fake_kwh_samples, axis=0)

    MAE_median = float(np.mean(np.abs(y_fake_kwh_point - y_real_kwh)))
    RMSE_mean = float(np.sqrt(np.mean((y_fake_kwh_mean - y_real_kwh) ** 2)))
    PeakMAE = float(np.mean(np.abs(y_fake_kwh_point.max(axis=1) - y_real_kwh.max(axis=1))))

    real_flat = y_real_kwh.reshape(-1)
    fake_flat = y_fake_kwh_point.reshape(-1)
    lo, hi = float(real_flat.min()), float(real_flat.max())
    if hi <= lo:
        hi = lo + 1e-6
    edges = np.linspace(lo, hi, BINS + 1)
    p, _ = np.histogram(real_flat, bins=edges, density=True)
    q, _ = np.histogram(fake_flat, bins=edges, density=True)
    eps = 1e-8
    p = (p + eps) / (p + eps).sum()
    q = (q + eps) / (q + eps).sum()
    # KL histogram estimates marginal distribution mismatch from binned real/generated kWh; source: https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence
    KL_hist = float(np.sum(p * np.log(p / q)))

    dtw_idxs = rng_eval.choice(PLOT_N, size=DTW_N, replace=False)
    DTW_mean = float(np.mean([dtw_distance(y_real_kwh[i, :, 0], y_fake_kwh_point[i, :, 0]) for i in dtw_idxs]))

    FTSD = frechet_distance(window_feature_embed(y_real_kwh), window_feature_embed(y_fake_kwh_point))

    X = y_fake_kwh_samples[:, :, :, 0]
    Y = y_real_kwh[:, :, 0]
    q10 = np.quantile(X, 0.05, axis=0)
    q50 = np.quantile(X, 0.50, axis=0)
    q90 = np.quantile(X, 0.95, axis=0)
    # CRPS is approximated from quantile pinball losses; source: https://en.wikipedia.org/wiki/Continuous_ranked_probability_score
    CRPS = float(2.0 * np.mean([
        np.mean(pinball(Y, q10, 0.05)),
        np.mean(pinball(Y, q50, 0.50)),
        np.mean(pinball(Y, q90, 0.95)),
    ]))
    # QuantileLoss averages pinball losses at the requested forecast quantiles; source: https://en.wikipedia.org/wiki/Quantile_regression
    QuantileLoss = float(np.mean([
        np.mean(pinball(Y, q10, 0.05)),
        np.mean(pinball(Y, q50, 0.50)),
        np.mean(pinball(Y, q90, 0.95)),
    ]))
    width = q90 - q10
    below = Y < q10
    above = Y > q90
    # Winkler interval score rewards narrow prediction intervals but penalises missed coverage; source: https://epiforecasts.io/scoringutils/articles/scoring-rules.html
    Winkler = float(np.mean(width + (2 / ALPHA_PI) * (q10 - Y) * below + (2 / ALPHA_PI) * (Y - q90) * above))
    Coverage = float(np.mean((Y >= q10) & (Y <= q90)))
    ACF_error = float(np.mean(np.abs(mean_acf(y_real_kwh) - mean_acf(y_fake_kwh_point))))

    scale = float(np.mean(y_real_kwh) + 1e-6)
    peak_scale = float(np.mean(y_real_kwh.max(axis=1)) + 1e-6)
    # CompositeScore is a project-specific tuning heuristic, not a standard literature metric.
    CompositeScore = float(np.mean([
        MAE_median / scale,
        RMSE_mean / scale,
        PeakMAE / peak_scale,
        CRPS / scale,
        QuantileLoss / scale,
        Winkler / scale,
        ACF_error,
        KL_hist,
        DTW_mean / (SEQ_LEN * scale),
        min(FTSD, 1e6) / (1.0 + min(FTSD, 1e6)),
    ]))

    corr_real = np.nan
    corr_fake = np.nan
    if "Temperature" in TIME_COV_COLS:
        temp = x_time_future_in[:, :, TIME_COV_COLS.index("Temperature")].reshape(-1)
        real_y = y_real_kwh.reshape(-1)
        fake_y = y_fake_kwh_point.reshape(-1)
        def corr(a, b):
            a = np.asarray(a)
            b = np.asarray(b)
            a = a - a.mean()
            b = b - b.mean()
            return float(np.dot(a, b) / (np.sqrt(np.dot(a, a)) * np.sqrt(np.dot(b, b)) + 1e-12))
        corr_real = corr(temp, real_y)
        corr_fake = corr(temp, fake_y)

    print("=" * 60)
    print(f"[{MODEL_LABEL}] forecast eval | pool={EVAL_POOL} | freq={FREQ} | horizon={PRIMARY_HORIZON} | context={CONTEXT_LEN} | seq_len={SEQ_LEN}")
    print(f"KL_hist={KL_hist:.6f} | DTW_mean={DTW_mean:.6f} | FTSD={FTSD:.6f}")
    print(f"MAE={MAE_median:.6f} | RMSE={RMSE_mean:.6f} | PeakMAE={PeakMAE:.6f}")
    print(f"CRPS={CRPS:.6f} | QuantileLoss={QuantileLoss:.6f} | Winkler={Winkler:.6f} | Coverage={Coverage:.4f}")
    print(f"ACF_error={ACF_error:.6f} | CompositeScore={CompositeScore:.6f}")

    return locals()

eval_results = {}
for _pool in EVAL_POOLS:
    eval_results[_pool] = run_eval_pool(_pool)

In [ ]:

# Embedded evaluation export helper. Kept local so server scripts do not need package imports.
import json
import math
import re
import zipfile
from datetime import datetime
from pathlib import Path
from xml.sax.saxutils import escape

import numpy as np


METRIC_KEYS = [
    "KL_hist",
    "DTW_mean",
    "FTSD",
    "MAE_median",
    "RMSE_mean",
    "PeakMAE",
    "CRPS",
    "QuantileLoss",
    "Winkler",
    "Coverage",
    "ACF_error",
    "CompositeScore",
    "corr_real",
    "corr_fake",
]

CONFIG_KEYS = [
    "MODEL_LABEL",
    "MODEL_TAG",
    "FREQ",
    "PRIMARY_HORIZON",
    "SEQ_LEN",
    "CONTEXT_LEN",
    "WINDOW_STRIDE",
    "WINDOW_STRIDE_MULTIPLIER",
    "VAL_FRAC",
    "SEED",
    "BATCH_SIZE",
    "LR",
    "SAMPLES",
    "PLOT_N",
    "EVAL_BATCH_SIZE",
    "EVAL_SEED",
    "EVAL_POOLS",
    "ALPHA_PI",
    "BINS",
    "CLIP_QUANTILE",
    "DTW_N",
    "epochs_diff",
    "cond_dim",
    "hidden_dim",
    "latent_dim",
    "diffusion_steps",
    "beta_start",
    "beta_end",
]


# Sanitises checkpoint names so export folder names are portable across Windows/Linux.
def _safe_stem(name: object) -> str:
    stem = Path(str(name)).name
    if stem.endswith(".pt"):
        stem = stem[:-3]
    stem = re.sub(r"[^A-Za-z0-9._=-]+", "_", stem)
    return stem or "evaluation"


def _as_excel_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (list, tuple, np.ndarray)):
        return json.dumps(np.asarray(value).tolist())
    if value is None:
        return ""
    if isinstance(value, float) and not math.isfinite(value):
        return str(value)
    return value


def _col_name(idx: int) -> str:
    name = ""
    while idx:
        idx, rem = divmod(idx - 1, 26)
        name = chr(65 + rem) + name
    return name


def _cell_xml(row_idx: int, col_idx: int, value) -> str:
    ref = f"{_col_name(col_idx)}{row_idx}"
    value = _as_excel_value(value)
    if isinstance(value, bool):
        return f'<c r="{ref}" t="b"><v>{1 if value else 0}</v></c>'
    if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value)):
        return f'<c r="{ref}"><v>{value}</v></c>'
    text = escape(str(value))
    return f'<c r="{ref}" t="inlineStr"><is><t>{text}</t></is></c>'


# Writes a minimal Excel workbook without adding an openpyxl dependency.
def write_metrics_xlsx(path: Path, rows: list[list[object]]) -> None:
    """Write a tiny .xlsx using only the standard library."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    sheet_rows = []
    for r_idx, row in enumerate(rows, start=1):
        cells = "".join(_cell_xml(r_idx, c_idx, value) for c_idx, value in enumerate(row, start=1))
        sheet_rows.append(f'<row r="{r_idx}">{cells}</row>')

    worksheet = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
        '<sheetData>'
        + "".join(sheet_rows)
        + '</sheetData></worksheet>'
    )

    workbook = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" '
        'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">'
        '<sheets><sheet name="metrics" sheetId="1" r:id="rId1"/></sheets></workbook>'
    )

    content_types = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
        '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
        '<Default Extension="xml" ContentType="application/xml"/>'
        '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>'
        '<Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>'
        '</Types>'
    )

    root_rels = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
        '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>'
        '</Relationships>'
    )

    workbook_rels = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
        '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/>'
        '</Relationships>'
    )

    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("[Content_Types].xml", content_types)
        zf.writestr("_rels/.rels", root_rels)
        zf.writestr("xl/workbook.xml", workbook)
        zf.writestr("xl/_rels/workbook.xml.rels", workbook_rels)
        zf.writestr("xl/worksheets/sheet1.xml", worksheet)


# Drops NaN/Inf values before histogram, QQ, and residual calculations.
def _finite_flat(arr) -> np.ndarray:
    flat = np.asarray(arr, dtype=float).reshape(-1)
    return flat[np.isfinite(flat)]


def _autocorr(x, max_lag=96):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.dot(x, x) + 1e-12
    acf = [1.0]
    for lag in range(1, max_lag + 1):
        acf.append(np.dot(x[:-lag], x[lag:]) / denom)
    return np.array(acf)


def _power_spectrum(x):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    fft = np.fft.rfft(x)
    ps = np.abs(fft) ** 2
    return ps / (ps.sum() + 1e-12)


def _save_fig(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return str(path)


# Creates the common diagnostic plot set used for visual model comparison.
def save_common_plots(
    out_dir: Path,
    model_label: str,
    y_real_kwh,
    y_fake_kwh_point,
    y_fake_kwh_samples=None,
    bins=80,
    clip_quantile=None,
    interval_alpha=0.10,
):
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt

    out_dir = Path(out_dir)
    plot_paths = []

    real_arr = np.asarray(y_real_kwh, dtype=float)
    fake_arr = np.asarray(y_fake_kwh_point, dtype=float)
    sample_arr = None
    if y_fake_kwh_samples is not None:
        sample_arr = np.asarray(y_fake_kwh_samples, dtype=float)
        if sample_arr.ndim != 4:
            sample_arr = None

    real = _finite_flat(real_arr)
    fake = _finite_flat(fake_arr)

    if clip_quantile is not None:
        cap = np.quantile(real, clip_quantile)
        real = np.clip(real, 0, cap)
        fake = np.clip(fake, 0, cap)

    lo, hi = float(real.min()), float(real.max())
    if hi <= lo:
        hi = lo + 1e-6
    edges = np.linspace(lo, hi, int(bins) + 1)

    real_y = real_arr[:, :, 0]
    fake_y = fake_arr[:, :, 0]
    t = np.arange(real_y.shape[1])

    real_mean = np.nanmean(real_y, axis=0)
    real_lo, real_hi = np.nanquantile(real_y, [0.10, 0.90], axis=0)
    if sample_arr is not None:
        fake_profile_pool = sample_arr[:, :, :, 0].reshape(-1, sample_arr.shape[2])
    else:
        fake_profile_pool = fake_y
    fake_mean = np.nanmean(fake_profile_pool, axis=0)
    fake_lo, fake_hi = np.nanquantile(fake_profile_pool, [0.10, 0.90], axis=0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(t, real_mean, label="Real mean", linewidth=2)
    ax.fill_between(t, real_lo, real_hi, alpha=0.18, label="Real 10-90% band")
    ax.plot(t, fake_mean, label="Generated mean", linewidth=2)
    ax.fill_between(t, fake_lo, fake_hi, alpha=0.18, label="Generated 10-90% band")
    ax.set_title(f"Multi-window Real vs Generated Profile | {model_label}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("kWh")
    ax.legend(loc="upper right")
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "01_multi_window_profile_bands.png"))
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(real, bins=edges, density=True, alpha=0.6, label="Real")
    ax.hist(fake, bins=edges, density=True, alpha=0.6, label="Generated")
    ax.set_title(f"All-window kWh Distribution | {model_label}")
    ax.set_xlabel("kWh")
    ax.set_ylabel("Density")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "02_probability_distribution.png"))
    plt.close(fig)

    q = np.linspace(0.01, 0.99, 300)
    real_q = np.quantile(real, q)
    fake_q = np.quantile(fake, q)
    mx = float(max(real_q.max(), fake_q.max(), 1e-6))
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(real_q, fake_q, marker=".", linestyle="none", alpha=0.6)
    ax.plot([0, mx], [0, mx], linewidth=1)
    ax.set_title(f"All-window QQ Plot | {model_label}")
    ax.set_xlabel("Real quantiles")
    ax.set_ylabel("Generated quantiles")
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "03_qq_plot.png"))
    plt.close(fig)

    max_lag = min(200, real_y.shape[1] - 1)
    acf_real = np.mean([_autocorr(real_y[i], max_lag) for i in range(real_y.shape[0])], axis=0)
    acf_fake = np.mean([_autocorr(fake_y[i], max_lag) for i in range(fake_y.shape[0])], axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(acf_real, label="Real ACF")
    ax.plot(acf_fake, label="Generated ACF")
    ax.set_title(f"Autocorrelation | {model_label}")
    ax.set_xlabel("Lag")
    ax.set_ylabel("ACF")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "04_autocorrelation.png"))
    plt.close(fig)

    ps_real = np.mean([_power_spectrum(real_y[i]) for i in range(real_y.shape[0])], axis=0)
    ps_fake = np.mean([_power_spectrum(fake_y[i]) for i in range(fake_y.shape[0])], axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ps_real, label="Real power spectrum")
    ax.plot(ps_fake, label="Generated power spectrum")
    ax.set_title(f"Power Spectrum | {model_label}")
    ax.set_xlabel("Frequency bin")
    ax.set_ylabel("Normalized power")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "05_power_spectrum.png"))
    plt.close(fig)

    resid = _finite_flat(fake_arr - real_arr)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(resid, bins=100, density=True, alpha=0.8)
    ax.set_title(f"Residual Distribution: Generated - Real | {model_label}")
    ax.set_xlabel("kWh error")
    ax.set_ylabel("Density")
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "06_residual_distribution.png"))
    plt.close(fig)

    true_vals = _finite_flat(real_arr)
    err_abs = _finite_flat(np.abs(fake_arr - real_arr))
    n = min(true_vals.shape[0], err_abs.shape[0])
    true_vals = true_vals[:n]
    err_abs = err_abs[:n]
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(true_vals[::50], err_abs[::50], alpha=0.3, s=5)
    ax.set_title(f"|Error| vs True kWh | {model_label}")
    ax.set_xlabel("True kWh")
    ax.set_ylabel("|Generated - Real|")
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "07_error_vs_true.png"))
    plt.close(fig)

    abs_err = np.abs(fake_y - real_y)
    window_mae = np.nanmean(abs_err, axis=1)
    window_rmse = np.sqrt(np.nanmean((fake_y - real_y) ** 2, axis=1))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(window_mae[np.isfinite(window_mae)], bins=60, alpha=0.7, label="Window MAE")
    ax.hist(window_rmse[np.isfinite(window_rmse)], bins=60, alpha=0.5, label="Window RMSE")
    ax.set_title(f"Window-level Error Distribution | {model_label}")
    ax.set_xlabel("kWh error")
    ax.set_ylabel("Window count")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "08_window_error_distribution.png"))
    plt.close(fig)

    real_mean_w = np.nanmean(real_y, axis=1)
    fake_mean_w = np.nanmean(fake_y, axis=1)
    real_peak_w = np.nanmax(real_y, axis=1)
    fake_peak_w = np.nanmax(fake_y, axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, x_vals, y_vals, label in [
        (axes[0], real_mean_w, fake_mean_w, "Window mean kWh"),
        (axes[1], real_peak_w, fake_peak_w, "Window peak kWh"),
    ]:
        ax.scatter(x_vals, y_vals, alpha=0.35, s=12)
        finite = np.isfinite(x_vals) & np.isfinite(y_vals)
        if np.any(finite):
            lim = float(max(np.nanmax(x_vals[finite]), np.nanmax(y_vals[finite]), 1e-6))
            ax.plot([0, lim], [0, lim], linewidth=1)
        ax.set_title(label)
        ax.set_xlabel("Real")
        ax.set_ylabel("Generated")
    fig.suptitle(f"Window Summary Agreement | {model_label}")
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "09_window_summary_scatter.png"))
    plt.close(fig)

    med_ae = np.nanmedian(abs_err, axis=0)
    lo_ae, hi_ae = np.nanquantile(abs_err, [0.10, 0.90], axis=0)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, med_ae, label="Median absolute error")
    ax.fill_between(t, lo_ae, hi_ae, alpha=0.2, label="10-90% absolute error band")
    ax.set_title(f"Error by Timestep Across Windows | {model_label}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("kWh absolute error")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "10_error_by_timestep.png"))
    plt.close(fig)

    resid_matrix = fake_y - real_y
    order = np.argsort(real_mean_w)
    vmax = np.nanquantile(np.abs(resid_matrix), 0.99)
    vmax = float(vmax if np.isfinite(vmax) and vmax > 0 else 1.0)
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(resid_matrix[order], aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(f"Residual Heatmap Across Windows | {model_label}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Validation windows sorted by real mean kWh")
    fig.colorbar(im, ax=ax, label="Generated - Real kWh")
    fig.tight_layout()
    plot_paths.append(_save_fig(fig, out_dir / "11_residual_heatmap.png"))
    plt.close(fig)

    if sample_arr is not None:
        alpha = float(interval_alpha)
        lo_pi = np.nanquantile(sample_arr[:, :, :, 0], alpha / 2, axis=0)
        hi_pi = np.nanquantile(sample_arr[:, :, :, 0], 1 - alpha / 2, axis=0)
        coverage_t = np.nanmean((real_y >= lo_pi) & (real_y <= hi_pi), axis=0)
        width_t = np.nanmean(hi_pi - lo_pi, axis=0)
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
        axes[0].plot(t, coverage_t)
        axes[0].axhline(1 - alpha, linewidth=1, linestyle="--")
        axes[0].set_title(f"90% Prediction Interval Coverage by Timestep | {model_label}")
        axes[0].set_ylabel("Coverage")
        axes[1].plot(t, width_t)
        axes[1].set_title("Mean Prediction Interval Width")
        axes[1].set_xlabel("Timestep")
        axes[1].set_ylabel("kWh")
        fig.tight_layout()
        plot_paths.append(_save_fig(fig, out_dir / "12_interval_coverage_by_timestep.png"))
        plt.close(fig)

    return plot_paths


# Flattens run configuration and metric values into the workbook row format.
def collect_common_metrics(namespace: dict, model_label: str, checkpoint_name: str):
    rows = [
        ["field", "value"],
        ["model", model_label],
        ["checkpoint_name", checkpoint_name],
        ["checkpoint_path", namespace.get("CHECKPOINT_PATH", "")],
        ["exported_at", datetime.now().isoformat(timespec="seconds")],
    ]

    for key in CONFIG_KEYS:
        if key in namespace:
            rows.append([key, _as_excel_value(namespace[key])])

    if "quantiles" in namespace:
        rows.append(["quantiles", _as_excel_value(namespace["quantiles"])])
    if "ALPHA_PI" in namespace:
        rows.append(["interval_percent", (1.0 - float(namespace["ALPHA_PI"])) * 100.0])

    rows.append(["", ""])
    rows.append(["metric", "value"])
    for key in METRIC_KEYS:
        if key in namespace:
            rows.append([key, _as_excel_value(namespace[key])])

    if "corr_real" in namespace and "corr_fake" in namespace:
        rows.append(["corr_abs_gap", abs(float(namespace["corr_real"]) - float(namespace["corr_fake"]))])

    return rows


# Writes metrics.xlsx and the shared PNG diagnostics for generative model outputs.
def export_common_evaluation(
    namespace: dict,
    model_label: str,
    checkpoint_name: str | None = None,
    output_root: str | Path = "evaluation_exports",
):
    checkpoint_name = checkpoint_name or namespace.get("CHECKPOINT_NAME") or model_label
    out_dir = Path(output_root) / _safe_stem(checkpoint_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = collect_common_metrics(namespace, model_label=model_label, checkpoint_name=str(checkpoint_name))
    metrics_path = out_dir / "metrics.xlsx"
    write_metrics_xlsx(metrics_path, rows)

    plots_dir = out_dir / "plots"
    bins = namespace.get("BINS", 80)
    clip_quantile = namespace.get("CLIP_QUANTILE", None)
    plot_paths = save_common_plots(
        plots_dir,
        model_label,
        namespace["y_real_kwh"],
        namespace["y_fake_kwh_point"],
        y_fake_kwh_samples=namespace.get("y_fake_kwh_samples"),
        bins=bins,
        clip_quantile=clip_quantile,
        interval_alpha=namespace.get("ALPHA_PI", 0.10),
    )

    return out_dir, metrics_path, plot_paths


for eval_pool, namespace in eval_results.items():
    namespace["CHECKPOINT_PATH"] = str(checkpoint_path)
    namespace["CHECKPOINT_NAME"] = checkpoint_name
    export_dir, metrics_path, plot_paths = export_common_evaluation(
        namespace,
        model_label=MODEL_LABEL,
        checkpoint_name=checkpoint_name,
        output_root=Path.cwd() / "evaluation_exports" / f"{eval_pool}_evaluation_exports",
    )
    print(f"Exported {eval_pool}: {export_dir}")
    print(f"Metrics: {metrics_path}")
    print(f"Plots: {len(plot_paths)}")